# Monitoring Dashboard - Pipeline Health

**Metrics tracked:**
- Processing latency
- Error rate
- Data quality (missing, duplicates)
- System resources (CPU, RAM)
- Model accuracy

---

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import json
from pathlib import Path
from datetime import datetime
import psutil
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

METRICS_DIR = Path('monitoring/metrics')
ALERTS_DIR = Path('monitoring/alerts')
METRICS_DIR.mkdir(parents=True, exist_ok=True)
ALERTS_DIR.mkdir(parents=True, exist_ok=True)

## 2. System Metrics

In [ ]:
# Get current system metrics
cpu_percent = psutil.cpu_percent(interval=1)
memory = psutil.virtual_memory()
disk = psutil.disk_usage('C:\\')

print('💻 SYSTEM RESOURCES')
print(f'CPU: {cpu_percent:.1f}%')
print(f'Memory: {memory.percent:.1f}% ({memory.used/1024**3:.1f} GB / {memory.total/1024**3:.1f} GB)')
print(f'Disk: {disk.percent:.1f}% ({disk.used/1024**3:.1f} GB / {disk.total/1024**3:.1f} GB)')

## 3. Data Quality Checks

In [ ]:
# Load cleaned data
df = pd.read_csv('data_clean/events_clean.csv', nrows=10000)

# Missing values
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'column': missing.index, 'missing_count': missing.values, 'missing_pct': missing_pct.values})
missing_df = missing_df[missing_df['missing_count'] > 0].sort_values('missing_pct', ascending=False)

print('📊 DATA QUALITY - Missing Values')
if len(missing_df) > 0:
    print(missing_df.to_string(index=False))
else:
    print('No missing values found!')

In [ ]:
# Duplicate check
dup_count = df.duplicated().sum()
dup_pct = dup_count / len(df) * 100
print(f'\n📊 DATA QUALITY - Duplicates')
print(f'Duplicate rows: {dup_count:,} ({dup_pct:.2f}%)')

## 4. Pipeline Metrics

In [ ]:
# Check if metrics exist
etl_metrics_file = METRICS_DIR / 'etl_pipeline_metrics.json'

if etl_metrics_file.exists():
    with open(etl_metrics_file) as f:
        etl_metrics = json.load(f)
    
    etl_df = pd.DataFrame(etl_metrics)
    print(f'ETL Pipeline - Total Runs: {len(etl_df)}')
    print(f'Success Rate: {(etl_df["status"] == "success").mean() * 100:.1f}%')
    if 'duration_seconds' in etl_df.columns:
        print(f'Avg Duration: {etl_df["duration_seconds"].mean():.1f}s')
else:
    print('No ETL metrics found. Run the pipeline first!')

## 5. Model Performance

In [ ]:
# Check model registry
registry_file = Path('models/model_registry.json')

if registry_file.exists():
    with open(registry_file) as f:
        registry = json.load(f)
    
    print('🤖 TRAINED MODELS')
    for model in registry['models']:
        print(f'\n  {model["name"]}')
        print(f'    Type: {model["type"]}')
        print(f'    Algorithm: {model["algorithm"]}')
        if 'metrics' in model:
            for metric, value in model['metrics'].items():
                print(f'    {metric}: {value:.4f}')
else:
    print('No models found. Run ML pipeline first!')

## 6. Alerts

In [ ]:
# Load recent alerts
alerts = []
for alert_file in sorted(ALERTS_DIR.glob('*.json'), reverse=True)[:10]:
    with open(alert_file) as f:
        alerts.append(json.load(f))

if alerts:
    print('🚨 RECENT ALERTS')
    for alert in alerts:
        severity = alert.get('severity', 'info').upper()
        print(f'  [{severity}] {alert.get("message", "")}')
else:
    print('No alerts! System healthy.')

## 7. Summary Dashboard

In [ ]:
print('\n' + '=' * 60)
print('📊 PIPELINE MONITORING DASHBOARD')
print('=' * 60)

print(f'\n⏰ Generated at: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')

print(f'\n💻 SYSTEM')
print(f'   CPU: {cpu_percent:.1f}%')
print(f'   Memory: {memory.percent:.1f}%')
print(f'   Disk: {disk.percent:.1f}%')

print(f'\n📦 ETL PIPELINE')
if etl_metrics_file.exists():
    print(f'   Runs: {len(etl_df)}')
    print(f'   Success: {(etl_df["status"] == "success").sum()}')
else:
    print('   No data')

print(f'\n🤖 ML MODELS')
if registry_file.exists():
    print(f'   Total models: {len(registry["models"])}')
else:
    print('   No models')

print(f'\n🚨 ALERTS')
print(f'   Recent alerts: {len(alerts)}')

print('\n' + '=' * 60)